# TFM · Predicción de Consumo y Precio Eléctrico — Madrid
## Notebook 04 · Modelado (v2 — corregido)

**Correcciones respecto a v1:**
- ✅ Data leakage eliminado (`precio_eur_mwh`, `precio_eur_kwh`, `demanda_mw` fuera de features)
- ✅ Evaluación igualada: todos los modelos sobre el mismo subconjunto del test
- ✅ XGBoost con early stopping usando validación
- ✅ LSTM mejorado: lookback=168h, loss=Huber, arquitectura más profunda
- ✅ Split de horizontes: modelo corto (h+1…h+12) y largo (h+13…h+48)

**Modelos:** SARIMA → XGBoost → LSTM

**Variables target:** `precio_eur_mwh` y `demanda_mw`

**Horizonte:** 48 horas

---
## 0. Setup

In [ ]:
!pip install xgboost statsmodels scikit-learn tensorflow shap --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import json, pickle, warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.multioutput import MultiOutputRegressor
import xgboost as xgb
import shap

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

plt.rcParams.update({
    'figure.figsize':    (14, 4),
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'font.size':         11,
})
COLORS   = ['#2563EB', '#F59E0B', '#10B981', '#EF4444', '#8B5CF6']
HORIZONTE = 48
H_CORTO   = 12   # Horizontes cortos: h+1 … h+12
H_LARGO   = 48   # Horizontes largos: h+13 … h+48
LOOKBACK  = 168  # Una semana de contexto para LSTM

tf.random.set_seed(42)
np.random.seed(42)
print('✅ Setup completado')
print(f'   TensorFlow: {tf.__version__} | XGBoost: {xgb.__version__}')

---
## 1. Carga de datos y corrección de data leakage

In [ ]:
with open('data/models/metadata.json') as f:
    meta = json.load(f)

with open('data/models/scaler_features.pkl', 'rb') as f:
    scaler_features = pickle.load(f)
with open('data/models/scaler_precio.pkl', 'rb') as f:
    scaler_precio = pickle.load(f)
with open('data/models/scaler_demanda.pkl', 'rb') as f:
    scaler_demanda = pickle.load(f)

def cargar(path):
    df = pd.read_csv(path)
    df['datetime'] = pd.to_datetime(df['datetime'], utc=True).dt.tz_convert('Europe/Madrid')
    return df.set_index('datetime').sort_index()

train        = cargar('data/processed/train.csv')
val          = cargar('data/processed/val.csv')
test         = cargar('data/processed/test.csv')
train_scaled = cargar('data/processed/train_scaled.csv')
val_scaled   = cargar('data/processed/val_scaled.csv')
test_scaled  = cargar('data/processed/test_scaled.csv')

TARGET_PRECIO  = meta['target_precio']
TARGET_DEMANDA = meta['target_demanda']

# ── CORRECCIÓN: eliminar leakage de FEATURE_COLS ──────────────────────────────
# precio_eur_mwh, precio_eur_kwh y demanda_mw son los propios targets
# Usarlos como features directas es data leakage — el modelo vería el futuro
LEAKAGE_COLS = ['precio_eur_mwh', 'precio_eur_kwh', 'demanda_mw']
FEATURE_COLS = [
    c for c in meta['feature_cols']
    if c not in LEAKAGE_COLS
]

print(f'✅ Features originales:  {len(meta["feature_cols"])}')
print(f'   Columnas con leakage eliminadas: {LEAKAGE_COLS}')
print(f'   Features limpias:     {len(FEATURE_COLS)}')
print(f'\nTrain: {train.shape} | Val: {val.shape} | Test: {test.shape}')

---
## 2. Funciones de evaluación

In [ ]:
def mape(y_true, y_pred):
    mask = np.abs(y_true) > 1.0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def evaluar(y_true, y_pred, nombre='Modelo', variable='precio'):
    mae_global  = mean_absolute_error(y_true, y_pred)
    rmse_global = np.sqrt(mean_squared_error(y_true, y_pred))
    mape_global = mape(y_true.ravel(), y_pred.ravel())
    mae_h  = [mean_absolute_error(y_true[:, h], y_pred[:, h]) for h in range(y_true.shape[1])]
    rmse_h = [np.sqrt(mean_squared_error(y_true[:, h], y_pred[:, h])) for h in range(y_true.shape[1])]
    unidad = '€/MWh' if variable == 'precio' else 'MW'
    print(f'\n{"="*52}')
    print(f'{nombre} — {variable}')
    print(f'{"="*52}')
    print(f'  MAE global:  {mae_global:>10.2f} {unidad}')
    print(f'  RMSE global: {rmse_global:>10.2f} {unidad}')
    print(f'  MAPE global: {mape_global:>10.2f}%')
    h_show = min(len(mae_h)-1, 47)
    print(f'  MAE  h+1: {mae_h[0]:.2f} | h+12: {mae_h[11]:.2f} | h+24: {mae_h[23]:.2f} | h+48: {mae_h[h_show]:.2f} {unidad}')
    return {'nombre': nombre, 'variable': variable,
            'mae': mae_global, 'rmse': rmse_global, 'mape': mape_global,
            'mae_h': mae_h, 'rmse_h': rmse_h}

def plot_prediccion(y_true, y_pred, nombre, variable, n_dias=7):
    horas = n_dias * 24
    real = y_true[:horas, 0]
    pred = y_pred[:horas, 0]
    fig, ax = plt.subplots(figsize=(16, 4))
    ax.plot(real, color=COLORS[0], lw=1.2, label='Real', alpha=0.9)
    ax.plot(pred, color=COLORS[3], lw=1.2, label='Predicción', alpha=0.9, ls='--')
    ax.set_title(f'{nombre} — {variable} — primeros {n_dias} días del test (h+1)', fontweight='bold')
    ax.set_xlabel('Horas')
    ax.set_ylabel('€/MWh' if variable == 'precio' else 'MW')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'data/processed/v2_pred_{nombre.lower().replace(" ","_")}_{variable}.png', dpi=150, bbox_inches='tight')
    plt.show()

def plot_mae_horizonte(resultados, variable):
    fig, ax = plt.subplots(figsize=(14, 4))
    for res, color in zip(resultados, COLORS):
        ax.plot(range(1, len(res['mae_h'])+1), res['mae_h'], lw=2, color=color, label=res['nombre'])
    ax.axvline(12, color='gray', ls=':', lw=0.9, alpha=0.7, label='h=12')
    ax.axvline(24, color='gray', ls='--', lw=0.9, alpha=0.7, label='h=24')
    ax.set_xlabel('Horizonte (horas)')
    ax.set_ylabel(f'MAE ({"€/MWh" if variable=="precio" else "MW"})')
    ax.set_title(f'MAE por horizonte — {variable}', fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'data/processed/v2_mae_horizonte_{variable}.png', dpi=150, bbox_inches='tight')
    plt.show()

print('✅ Funciones definidas')

---
## 3. Modelo 1 — SARIMA (Baseline)

Baseline univariante. Evaluado sobre el mismo subconjunto N_EVAL del test
que XGBoost y LSTM para comparación justa.

In [ ]:
# ── Test ADF ───────────────────────────────────────────────────────────────────
for variable, col in [('precio', 'precio_eur_mwh'), ('demanda', 'demanda_mw')]:
    adf = adfuller(train[col].dropna(), autolag='AIC')
    print(f'{variable}: p={adf[1]:.4f} → {"Estacionaria ✅" if adf[1]<0.05 else "No estacionaria ⚠️"}')

# ── Evaluación rolling — mismos N_EVAL puntos que ML/DL ───────────────────────
N_EVAL    = 100   # Puntos de evaluación comunes a todos los modelos
N_SARIMA  = 24 * 7 * 4  # Ventana de entrenamiento SARIMA: 4 semanas

serie_precio_full  = pd.concat([train['precio_eur_mwh'],  val['precio_eur_mwh'],  test['precio_eur_mwh']])
serie_demanda_full = pd.concat([train['demanda_mw'],       val['demanda_mw'],       test['demanda_mw']])
idx_test = len(train) + len(val)

def evaluar_sarima_rolling(serie_full, idx_start, n_eval=N_EVAL):
    preds, reales = [], []
    for i in range(n_eval):
        historia = serie_full.iloc[:idx_start + i]
        modelo   = SARIMAX(
            historia.iloc[-N_SARIMA:],
            order=(1,1,1), seasonal_order=(1,1,1,24),
            enforce_stationarity=False, enforce_invertibility=False
        ).fit(disp=False)
        pred_48 = modelo.forecast(steps=HORIZONTE).values
        real_48 = serie_full.iloc[idx_start+i : idx_start+i+HORIZONTE].values
        if len(real_48) == HORIZONTE:
            preds.append(pred_48)
            reales.append(real_48)
        if (i+1) % 25 == 0:
            print(f'  {i+1}/{n_eval}...')
    return np.array(reales), np.array(preds)

print('Evaluando SARIMA precio...')
y_true_sarima_p, y_pred_sarima_p = evaluar_sarima_rolling(serie_precio_full,  idx_test)
res_sarima_precio  = evaluar(y_true_sarima_p, y_pred_sarima_p, 'SARIMA', 'precio')

print('\nEvaluando SARIMA demanda...')
y_true_sarima_d, y_pred_sarima_d = evaluar_sarima_rolling(serie_demanda_full, idx_test)
res_sarima_demanda = evaluar(y_true_sarima_d, y_pred_sarima_d, 'SARIMA', 'demanda')

---
## 4. Modelo 2 — XGBoost con Early Stopping y Split de Horizontes

Dos modelos por variable:
- **Corto** (h+1…h+12): mayor precisión en predicciones inmediatas
- **Largo** (h+13…h+48): captura tendencia a medio plazo

Early stopping usando el conjunto de validación para evitar overfitting.

In [ ]:
# ── Matrices X / y sin leakage ────────────────────────────────────────────────
X_train = train_scaled[FEATURE_COLS].values
X_val   = val_scaled[FEATURE_COLS].values
X_test  = test_scaled[FEATURE_COLS].values

# Targets escalados
y_train_p = train_scaled[TARGET_PRECIO].values
y_val_p   = val_scaled[TARGET_PRECIO].values
y_test_p  = test_scaled[TARGET_PRECIO].values

y_train_d = train_scaled[TARGET_DEMANDA].values
y_val_d   = val_scaled[TARGET_DEMANDA].values
y_test_d  = test_scaled[TARGET_DEMANDA].values

# Split corto / largo en los índices de targets
IDX_CORTO = list(range(0, H_CORTO))            # h+1  … h+12  → índices 0..11
IDX_LARGO = list(range(H_CORTO, HORIZONTE))    # h+13 … h+48  → índices 12..47

print(f'Features: {X_train.shape[1]}')
print(f'Horizonte corto: h+1…h+{H_CORTO} ({len(IDX_CORTO)} outputs)')
print(f'Horizonte largo: h+{H_CORTO+1}…h+{HORIZONTE} ({len(IDX_LARGO)} outputs)')

In [ ]:
# ── XGBoost con early stopping por horizonte ──────────────────────────────────
# MultiOutputRegressor no soporta eval_set directamente,
# así que entrenamos un modelo por horizonte con early stopping
# y los agrupamos en una función de predicción.

def entrenar_xgb_con_es(
    X_tr, y_tr, X_vl, y_vl,
    n_estimators=1000, max_depth=6, lr=0.05, verbose_each=50
):
    """
    Entrena un XGBRegressor por horizonte con early stopping.
    Devuelve lista de modelos entrenados.
    """
    modelos = []
    n_horizontes = y_tr.shape[1]
    for h in range(n_horizontes):
        m = xgb.XGBRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=lr,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=3,
            reg_alpha=0.1,
            reg_lambda=1.0,
            random_state=42,
            n_jobs=-1,
            tree_method='hist',
            early_stopping_rounds=30,
            eval_metric='mae',
        )
        m.fit(
            X_tr, y_tr[:, h],
            eval_set=[(X_vl, y_vl[:, h])],
            verbose=False
        )
        modelos.append(m)
        if (h+1) % verbose_each == 0:
            print(f'  Horizonte {h+1}/{n_horizontes} — best_iteration: {m.best_iteration}')
    return modelos

def predecir_xgb(modelos, X):
    return np.column_stack([m.predict(X) for m in modelos])

# ── PRECIO ────────────────────────────────────────────────────────────────────
print('Entrenando XGBoost precio — horizonte corto (h+1…h+12)...')
xgb_precio_corto = entrenar_xgb_con_es(
    X_train, y_train_p[:, IDX_CORTO],
    X_val,   y_val_p[:, IDX_CORTO]
)

print('\nEntrenando XGBoost precio — horizonte largo (h+13…h+48)...')
xgb_precio_largo = entrenar_xgb_con_es(
    X_train, y_train_p[:, IDX_LARGO],
    X_val,   y_val_p[:, IDX_LARGO]
)

# ── DEMANDA ───────────────────────────────────────────────────────────────────
print('\nEntrenando XGBoost demanda — horizonte corto...')
xgb_demanda_corto = entrenar_xgb_con_es(
    X_train, y_train_d[:, IDX_CORTO],
    X_val,   y_val_d[:, IDX_CORTO]
)

print('\nEntrenando XGBoost demanda — horizonte largo...')
xgb_demanda_largo = entrenar_xgb_con_es(
    X_train, y_train_d[:, IDX_LARGO],
    X_val,   y_val_d[:, IDX_LARGO]
)
print('\n✅ XGBoost entrenado')

In [ ]:
# ── Predicción — combinar corto + largo ───────────────────────────────────────
# Usamos los mismos N_EVAL primeros puntos del test que SARIMA
X_test_eval = X_test[:N_EVAL]

pred_p_corto_sc = predecir_xgb(xgb_precio_corto,  X_test_eval)
pred_p_largo_sc = predecir_xgb(xgb_precio_largo,  X_test_eval)
pred_d_corto_sc = predecir_xgb(xgb_demanda_corto, X_test_eval)
pred_d_largo_sc = predecir_xgb(xgb_demanda_largo, X_test_eval)

# Reconstruir array completo (48 horizontes)
pred_xgb_p_sc = np.concatenate([pred_p_corto_sc, pred_p_largo_sc], axis=1)
pred_xgb_d_sc = np.concatenate([pred_d_corto_sc, pred_d_largo_sc], axis=1)

# Invertir escalado
pred_xgb_precio  = scaler_precio.inverse_transform(pred_xgb_p_sc)
pred_xgb_demanda = scaler_demanda.inverse_transform(pred_xgb_d_sc)
real_precio      = scaler_precio.inverse_transform(y_test_p[:N_EVAL])
real_demanda     = scaler_demanda.inverse_transform(y_test_d[:N_EVAL])

res_xgb_precio  = evaluar(real_precio,  pred_xgb_precio,  'XGBoost', 'precio')
res_xgb_demanda = evaluar(real_demanda, pred_xgb_demanda, 'XGBoost', 'demanda')

plot_prediccion(real_precio,  pred_xgb_precio,  'XGBoost', 'precio')
plot_prediccion(real_demanda, pred_xgb_demanda, 'XGBoost', 'demanda')

In [ ]:
# ── SHAP — modelo h+1 de precio (sin leakage) ─────────────────────────────────
modelo_h1 = xgb_precio_corto[0]  # h+1
explainer  = shap.TreeExplainer(modelo_h1)
shap_vals  = explainer.shap_values(X_test_eval)

plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_vals, X_test_eval,
    feature_names=FEATURE_COLS,
    max_display=20,
    show=False
)
plt.title('SHAP — XGBoost precio h+1 (sin leakage)', fontweight='bold')
plt.tight_layout()
plt.savefig('data/processed/v2_shap_precio.png', dpi=150, bbox_inches='tight')
plt.show()
print('Features más importantes (SHAP):')
imp = pd.Series(
    np.abs(shap_vals).mean(axis=0),
    index=FEATURE_COLS
).sort_values(ascending=False)
print(imp.head(10))

In [ ]:
for nombre, obj in [
    ('xgb_precio_corto',  xgb_precio_corto),
    ('xgb_precio_largo',  xgb_precio_largo),
    ('xgb_demanda_corto', xgb_demanda_corto),
    ('xgb_demanda_largo', xgb_demanda_largo),
]:
    with open(f'data/models/{nombre}.pkl', 'wb') as f:
        pickle.dump(obj, f)
print('✅ Modelos XGBoost guardados')

---
## 5. Modelo 3 — LSTM mejorado

Mejoras respecto a v1:
- **Lookback = 168h** (una semana): más contexto histórico
- **Loss = Huber**: más robusto ante outliers de precio
- **Arquitectura más profunda**: LSTM(256) → LSTM(128) → LSTM(64)
- **Dense(128)** antes de la capa de salida

In [ ]:
def crear_secuencias(X, y, lookback=LOOKBACK):
    X_seq, y_seq = [], []
    for i in range(lookback, len(X)):
        X_seq.append(X[i-lookback:i])
        y_seq.append(y[i])
    return np.array(X_seq, dtype=np.float32), np.array(y_seq, dtype=np.float32)

# Secuencias precio
X_tr_seq_p, y_tr_seq_p = crear_secuencias(X_train, y_train_p)
X_vl_seq_p, y_vl_seq_p = crear_secuencias(X_val,   y_val_p)
X_te_seq_p, y_te_seq_p = crear_secuencias(X_test,  y_test_p)

# Secuencias demanda
X_tr_seq_d, y_tr_seq_d = crear_secuencias(X_train, y_train_d)
X_vl_seq_d, y_vl_seq_d = crear_secuencias(X_val,   y_val_d)
X_te_seq_d, y_te_seq_d = crear_secuencias(X_test,  y_test_d)

n_features = X_tr_seq_p.shape[2]
print(f'Shape secuencias: {X_tr_seq_p.shape}  →  lookback={LOOKBACK}, features={n_features}')

In [ ]:
def construir_lstm(n_features, lookback=LOOKBACK, horizonte=HORIZONTE):
    inp = Input(shape=(lookback, n_features))
    x   = LSTM(256, return_sequences=True)(inp)
    x   = Dropout(0.2)(x)
    x   = LSTM(128, return_sequences=True)(x)
    x   = Dropout(0.2)(x)
    x   = LSTM(64,  return_sequences=False)(x)
    x   = Dense(128, activation='relu')(x)
    x   = Dropout(0.1)(x)
    out = Dense(horizonte)(x)
    model = Model(inp, out)
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss=tf.keras.losses.Huber(delta=1.0),  # Robusto ante outliers
        metrics=['mae']
    )
    return model

lstm_precio  = construir_lstm(n_features)
lstm_demanda = construir_lstm(n_features)
lstm_precio.summary()

In [ ]:
# ── Activar GPU si está disponible ────────────────────────────────────────────
print('GPU disponible:', tf.config.list_physical_devices('GPU'))

callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5, verbose=1)
]

print('\nEntrenando LSTM precio...')
hist_p = lstm_precio.fit(
    X_tr_seq_p, y_tr_seq_p,
    validation_data=(X_vl_seq_p, y_vl_seq_p),
    epochs=50, batch_size=64,
    callbacks=callbacks, verbose=1
)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(hist_p.history['loss'],     color=COLORS[0], label='Train')
ax.plot(hist_p.history['val_loss'], color=COLORS[3], label='Val')
ax.set_title('LSTM precio — curva de aprendizaje (Huber loss)', fontweight='bold')
ax.set_xlabel('Época'); ax.set_ylabel('Huber loss'); ax.legend()
plt.tight_layout()
plt.savefig('data/processed/v2_lstm_precio_learning.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('Entrenando LSTM demanda...')
hist_d = lstm_demanda.fit(
    X_tr_seq_d, y_tr_seq_d,
    validation_data=(X_vl_seq_d, y_vl_seq_d),
    epochs=50, batch_size=64,
    callbacks=callbacks, verbose=1
)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(hist_d.history['loss'],     color=COLORS[0], label='Train')
ax.plot(hist_d.history['val_loss'], color=COLORS[3], label='Val')
ax.set_title('LSTM demanda — curva de aprendizaje (Huber loss)', fontweight='bold')
ax.set_xlabel('Época'); ax.set_ylabel('Huber loss'); ax.legend()
plt.tight_layout()
plt.savefig('data/processed/v2_lstm_demanda_learning.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Evaluación LSTM sobre los mismos N_EVAL puntos ────────────────────────────
pred_lstm_p_sc = lstm_precio.predict(X_te_seq_p[:N_EVAL],  verbose=0)
pred_lstm_d_sc = lstm_demanda.predict(X_te_seq_d[:N_EVAL], verbose=0)

pred_lstm_precio  = scaler_precio.inverse_transform(pred_lstm_p_sc)
pred_lstm_demanda = scaler_demanda.inverse_transform(pred_lstm_d_sc)
real_precio_seq   = scaler_precio.inverse_transform(y_te_seq_p[:N_EVAL])
real_demanda_seq  = scaler_demanda.inverse_transform(y_te_seq_d[:N_EVAL])

res_lstm_precio  = evaluar(real_precio_seq,  pred_lstm_precio,  'LSTM', 'precio')
res_lstm_demanda = evaluar(real_demanda_seq, pred_lstm_demanda, 'LSTM', 'demanda')

plot_prediccion(real_precio_seq,  pred_lstm_precio,  'LSTM', 'precio')
plot_prediccion(real_demanda_seq, pred_lstm_demanda, 'LSTM', 'demanda')

lstm_precio.save('data/models/v2_lstm_precio.keras')
lstm_demanda.save('data/models/v2_lstm_demanda.keras')
print('✅ Modelos LSTM guardados')

---
## 6. Comparativa final

In [ ]:
print('='*72)
print(f'COMPARATIVA FINAL — evaluación sobre los mismos {N_EVAL} puntos del test')
print('='*72)

for titulo, resultados in [
    ('PRECIO (€/MWh)', [res_sarima_precio,  res_xgb_precio,  res_lstm_precio]),
    ('DEMANDA (MW)',   [res_sarima_demanda, res_xgb_demanda, res_lstm_demanda])
]:
    print(f'\n{titulo}')
    print(f'{"Modelo":<12}{"MAE":>10}{"RMSE":>10}{"MAPE%":>10}{"MAE h+1":>12}{"MAE h+24":>12}{"MAE h+48":>12}')
    print('-'*78)
    for r in resultados:
        h48 = r['mae_h'][47] if len(r['mae_h']) > 47 else r['mae_h'][-1]
        print(f'{r["nombre"]:<12}{r["mae"]:>10.2f}{r["rmse"]:>10.2f}{r["mape"]:>10.2f}'
              f'{r["mae_h"][0]:>12.2f}{r["mae_h"][23]:>12.2f}{h48:>12.2f}')

plot_mae_horizonte([res_sarima_precio,  res_xgb_precio,  res_lstm_precio],  'precio')
plot_mae_horizonte([res_sarima_demanda, res_xgb_demanda, res_lstm_demanda], 'demanda')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Comparativa MAE global ({N_EVAL} puntos test, sin leakage)', fontweight='bold')

modelos = ['SARIMA', 'XGBoost', 'LSTM']
for ax, (titulo, resultados) in zip(axes, [
    ('Precio (€/MWh)', [res_sarima_precio,  res_xgb_precio,  res_lstm_precio]),
    ('Demanda (MW)',   [res_sarima_demanda, res_xgb_demanda, res_lstm_demanda])
]):
    maes = [r['mae'] for r in resultados]
    bars = ax.bar(modelos, maes, color=COLORS[:3], alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, maes):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01,
                f'{val:.2f}', ha='center', fontsize=10, fontweight='bold')
    idx_mejor = int(np.argmin(maes))
    bars[idx_mejor].set_edgecolor('black')
    bars[idx_mejor].set_linewidth(2.5)
    ax.set_title(titulo); ax.set_ylabel('MAE')

plt.tight_layout()
plt.savefig('data/processed/v2_comparativa_mae.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
mejor_p = min([res_sarima_precio,  res_xgb_precio,  res_lstm_precio],  key=lambda x: x['mae'])
mejor_d = min([res_sarima_demanda, res_xgb_demanda, res_lstm_demanda], key=lambda x: x['mae'])

mejora_p = (res_sarima_precio['mae']  - mejor_p['mae']) / res_sarima_precio['mae']  * 100
mejora_d = (res_sarima_demanda['mae'] - mejor_d['mae']) / res_sarima_demanda['mae'] * 100

print('='*60)
print('CONCLUSIONES (v2 — sin leakage, evaluación igualada)')
print('='*60)
print(f'''
PRECIO:
  Mejor modelo: {mejor_p["nombre"]}  (MAE = {mejor_p["mae"]:.2f} €/MWh)
  Mejora sobre SARIMA: {mejora_p:.1f}%

DEMANDA:
  Mejor modelo: {mejor_d["nombre"]}  (MAE = {mejor_d["mae"]:.2f} MW)
  Mejora sobre SARIMA: {mejora_d:.1f}%

DIFERENCIAS CLAVE RESPECTO A v1:
  - Eliminado data leakage: resultados ahora son honestos
  - Evaluación sobre el mismo subconjunto: comparación justa
  - XGBoost con early stopping: mejor generalización
  - LSTM con Huber loss: más robusto ante picos de precio
  - Split corto/largo en XGBoost: mejor calibración por horizonte
''')

# Guardar resultados
resultados_json = {}
for var, resList in [
    ('precio',  [res_sarima_precio,  res_xgb_precio,  res_lstm_precio]),
    ('demanda', [res_sarima_demanda, res_xgb_demanda, res_lstm_demanda])
]:
    resultados_json[var] = {}
    for r in resList:
        resultados_json[var][r['nombre'].lower()] = {
            'mae':    float(r['mae']),
            'rmse':   float(r['rmse']),
            'mape':   float(r['mape']),
            'mae_h':  [float(x) for x in r['mae_h']],
            'rmse_h': [float(x) for x in r['rmse_h']],
        }

with open('data/models/resultados_v2.json', 'w') as f:
    json.dump(resultados_json, f, indent=2)

print('✅ Notebook 04 v2 completado.')
print('   Resultados guardados en data/models/resultados_v2.json')